# Week 3, day 1 (afternoon) — Worksheet 17 SOLUTIONS: bronze, silver, gold

Executed in the lab image against the real superstore extract. Every quoted
number is what it actually printed.

Question 2 is the one that justifies the whole architecture: the duplicate rows
are not corruption, they are a **re-delivery**, and telling those two apart is
what bronze is for.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 17 — Bronze, silver, gold. Run this once.
import glob
import pandas as pd

BRONZE = "data/bronze/"

print("the landing zone, as delivered:")
for path in sorted(glob.glob(BRONZE + "*.csv")):
    n = sum(1 for _ in open(path)) - 1
    print("  %-22s %5d rows" % (path.split("/")[-1], n))

BRONZE — take delivery, lose nothing

### Question 1

Build the bronze layer. Read every `orders_*.csv` and concatenate them, adding two lineage columns: `_source_file` (which file the row came from) and `_ingested_at` (a fixed timestamp). Print the shape and the row count per source file.
> **NOTE:** bronze does not clean. Do not deduplicate, do not fix types, do not join. Take delivery and record where each row came from.

In [ ]:
frames = []
for path in sorted(glob.glob(BRONZE + "orders_*.csv")):
    part = pd.read_csv(path)
    part["_source_file"] = path.split("/")[-1]
    part["_ingested_at"] = pd.Timestamp("2026-08-30 06:00:00")
    frames.append(part)

bronze_orders = pd.concat(frames, ignore_index=True)
print("bronze_orders:", bronze_orders.shape)
print()
print(bronze_orders.groupby("_source_file").size().rename("rows").to_string())
print()
print("total:", len(bronze_orders))

```
bronze_orders: (8100, 15)

_source_file
orders_2009.csv    2073
orders_2010.csv    2056
orders_2011.csv    1911
orders_2012.csv    2060

total: 8100
```

Four files, 8,100 rows, thirteen source columns plus two of our own.

The two added columns are the whole point of bronze. `_source_file` says **where
each row came from**; `_ingested_at` says **when it arrived**. Neither exists in
the source, and without them bronze is just a copy.

What they buy you is concrete:

**A bad load becomes reversible.** `bronze[bronze._source_file !=
'orders_2012.csv']` removes exactly one delivery. Without lineage the only way to
undo a load is to rebuild everything.

**A number becomes traceable.** When a figure looks wrong in a dashboard, the
trail runs gold → silver → bronze → *this file, ingested at this time* — and then
out to whoever sent it. Lineage is what stops the trail ending at your pipeline's
front door.

**Question 2 becomes answerable at all.** The duplicates are only diagnosable
because each copy remembers which file it came in.

Note what bronze deliberately does **not** do: no dedup, no type fixing, no
joins, no filtering. Every one of those is a decision, and decisions belong in a
layer you can rebuild. Bronze's only job is *nothing is lost*.

The per-file counts are worth printing every run. They are the cheapest possible
delivery check: 2012 has 2,060 rows where the other years have ~2,000, which is
not obviously wrong — but it is the thread question 2 pulls.

### Question 2

Bronze is append-only, so it can contain the same row twice. Find the duplicates: count rows whose `LineID` repeats, print which `_source_file` each copy came from, and the `OrderDate` range they cover.
> **NOTE:** before calling this corruption, look at *which files* the two copies arrived in.

In [ ]:
frames = []
for path in sorted(glob.glob(BRONZE + "orders_*.csv")):
    part = pd.read_csv(path)
    part["_source_file"] = path.split("/")[-1]
    frames.append(part)
bronze_orders = pd.concat(frames, ignore_index=True)

dupes = bronze_orders[bronze_orders.LineID.duplicated(keep=False)]
print("rows with a repeated LineID:", len(dupes))
print("distinct LineIDs involved:  ", dupes.LineID.nunique())
print()
print("which files the copies came from:")
print(dupes.groupby("_source_file").size().rename("rows").to_string())
print()
print("OrderDate range of the duplicated rows:",
      dupes.OrderDate.min(), "->", dupes.OrderDate.max())
print()
ignoring_lineage = dupes.drop(columns=["_source_file"])
print("copies identical apart from lineage:",
      int(ignoring_lineage.duplicated().sum()), "of", len(dupes))

```
rows with a repeated LineID: 80
distinct LineIDs involved:   40

which files the copies came from:
_source_file
orders_2011.csv    40
orders_2012.csv    40

OrderDate range of the duplicated rows: 2011-12-24 -> 2011-12-31

copies identical apart from lineage: 40 of 80
```

Read the middle block first, because it changes what this is.

**The two copies came from two different files** — 40 rows in `orders_2011.csv`
and the same 40 in `orders_2012.csv` — and they cover **2011-12-24 to
2011-12-31**, the last week of 2011.

That is not corruption. That is a **re-delivery**: the 2012 extract was run with
a start date a few days early and re-sent the tail of the previous partition.
Overlapping deliveries are routine — a backfill, a re-run after a failure, an
at-least-once pipeline, a source that defines "since" inclusively.

`copies identical apart from lineage: 40 of 80` confirms it. Every value matches;
only `_source_file` differs. Nothing was corrupted, edited, or lost — the same
rows simply arrived twice.

**This is exactly why bronze and silver are separate layers.** Consider the
alternatives:

- **Reject the 2012 file** as containing duplicates → you lose a year of real
  data because 40 rows overlapped.
- **Deduplicate on ingest** → bronze no longer matches what the source sent, and
  you can no longer prove what arrived or reproduce the problem.
- **Keep both in bronze, dedup on the way to silver** → bronze stays a faithful
  record, silver is correct, and the overlap is diagnosable forever.

Only the third preserves both properties. Bronze's promise is *nothing is lost*,
not *everything is right*.

And the distinction matters for what you do next. A **re-delivery** is handled
silently by a dedup rule. **Genuinely conflicting copies** — same key, different
values — are an escalation, because somebody has to decide which is true. The
`identical apart from lineage` check is what tells the two apart, and it costs one
line.

SILVER — one row means one thing

### Question 3

Deduplicate into the silver grain. Keep the **first** copy of each `LineID`, and print the row count before and after, plus `SUM(Sales)` before and after.
> **NOTE:** the difference in `SUM(Sales)` is what the re-delivery would have added to revenue.

In [ ]:
frames = [pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))]
bronze_orders = pd.concat(frames, ignore_index=True)

silver = bronze_orders.drop_duplicates(subset="LineID", keep="first").copy()
print("bronze rows:", len(bronze_orders))
print("silver rows:", len(silver))
print("dropped:    ", len(bronze_orders) - len(silver))
print()
print("SUM(Sales) bronze: %14.2f" % bronze_orders.Sales.sum())
print("SUM(Sales) silver: %14.2f" % silver.Sales.sum())
print("overstated by:     %14.2f  (%.3f%%)"
      % (bronze_orders.Sales.sum() - silver.Sales.sum(),
         100 * (bronze_orders.Sales.sum() / silver.Sales.sum() - 1)))
print()
print("LineID unique now:", silver.LineID.is_unique)

```
bronze rows: 8100
silver rows: 8060
dropped:     40

SUM(Sales) bronze:    13633504.89
SUM(Sales) silver:    13570810.63
overstated by:           62694.27  (0.462%)

LineID unique now: True
```

Forty rows dropped, and **62,694.27 of revenue with them — 0.462%.**

That percentage is the reason this matters. It is not large enough to notice.
Revenue reported as 13.63M instead of 13.57M passes every smell test: it is the
right order of magnitude, it moves sensibly month over month, and no reconciliation
tolerance of 1% would flag it. It is simply wrong, quietly, forever, and it would
grow every time a delivery overlapped again.

`keep="first"` is a decision worth stating rather than defaulting into. Here the
copies are identical (question 2), so first/last are the same rows and it does not
matter. It matters enormously when they are **not** identical — a corrected
address, a restated amount — and then `keep="last"` (take the newest) is usually
right, but only if the files are processed in delivery order. If you cannot
guarantee that, sort by `_ingested_at` first and say so.

`LineID unique now: True` is the promise silver exists to make. From this line
onward, one row means one order line — so `COUNT(*)` counts lines, `SUM(Sales)`
totals revenue once, and a downstream join can be validated. None of those are
true of bronze, and none of them should be attempted there.

Question 10 is what happens when somebody attempts them anyway.

### Question 4

Conform it: join the customer and product dimensions onto silver with `how="left"` and `validate="many_to_one"`. Print the row count after each join and the columns gained.

In [ ]:
frames = [pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))]
silver = (pd.concat(frames, ignore_index=True)
            .drop_duplicates(subset="LineID", keep="first"))
customers = pd.read_csv(BRONZE + "customers.csv")
products = pd.read_csv(BRONZE + "products.csv")

before = len(silver)
silver = silver.merge(customers[["CustomerID", "CustomerName", "Province",
                                 "Region", "CustomerSegment"]],
                      on="CustomerID", how="left", validate="many_to_one")
print("+ customers  %5d -> %5d rows" % (before, len(silver)))

before = len(silver)
silver = silver.merge(products[["ProductID", "ProductName", "ProductCategory",
                                "ProductSubCategory"]],
                      on="ProductID", how="left", validate="many_to_one")
print("+ products   %5d -> %5d rows" % (before, len(silver)))
print()
print("grain preserved:", silver.LineID.is_unique)
print("columns:", len(silver.columns))

```
+ customers   8060 ->  8060 rows
+ products    8060 ->  8060 rows

grain preserved: True
columns: 20
```

Two joins, **zero rows added**, grain intact. That is conforming: the fact table
gains context and stays the same length.

`validate="many_to_one"` is doing real work here even though it passes silently.
It asserts `CustomerID` is unique in `customers` and `ProductID` is unique in
`products` — and if a dimension export ever double-delivered the way the orders
file did, this line fails loudly instead of inflating revenue by a few percent.
A dimension with a duplicated key is the single most common cause of a
mysteriously-slightly-too-high total.

Three deliberate choices in that code:

**Only the needed columns.** `customers[["CustomerID", "CustomerName", ...]]`
rather than the whole frame — so nothing collides (worksheet 16 question 9) and
the join documents what it is for.

**`how="left"`, always.** Silver's row count is defined by the fact table. An
inner join would let a missing dimension row silently delete order lines, and the
row count would go *down* — which looks like less data rather than a bug.

**Printed before and after.** `8060 -> 8060` is the assertion. Three characters
of output, and it is the difference between knowing and assuming.

Twenty columns now: thirteen from the source, five from customers, and the
product attributes. This is the table an analyst should actually be given —
worksheet 16 needed three joins to answer "revenue by segment"; from here it is a
`groupby`.

### Question 5

Add the return flag — **without** a join, using the lesson from worksheet 16 question 8. Print the row count (it must not change), the returned line count, and the returned order count.
> **NOTE:** `returns` is one row per order and silver is one row per line. A merge here fans out the returns side.

In [ ]:
frames = [pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))]
silver = (pd.concat(frames, ignore_index=True)
            .drop_duplicates(subset="LineID", keep="first").copy())
returns = pd.read_csv(BRONZE + "returns.csv")

before = len(silver)
silver["IsReturned"] = silver.OrderID.isin(set(returns.OrderID))
print("rows before: %d   after: %d   changed: %d"
      % (before, len(silver), len(silver) - before))
print()
print("returned LINE rows: ", int(silver.IsReturned.sum()))
print("returned ORDERS:    ", silver.loc[silver.IsReturned, "OrderID"].nunique())
print("returns table rows: ", len(returns))
print()
print("if we had merged instead:",
      len(silver.merge(returns, on="OrderID", how="inner")), "rows")

```
rows before: 8060   after: 8060   changed: 0

returned LINE rows:  837
returned ORDERS:     558
returns table rows:  558

if we had merged instead: 837 rows
```

The flag adds information without adding rows — **0 changed** — and the last line
shows what the obvious alternative would have cost: a merge produces **837 rows**
from 8,060, silently re-shaping silver.

This is worksheet 16 question 8 applied as a design rule. `returns` is one row
per *order*; silver is one row per *line*. Merging them fans out the returns side,
and while `SUM(Sales)` would survive that (each line still appears once), the row
count would not — and silver's entire promise is its row count.

**`isin` cannot break the grain.** It computes a boolean per existing row. There
is no version of it that adds or removes rows, so no future reader has to reason
about whether silver is still one-row-per-line.

The three numbers underneath are the reconciliation:

- **837 returned line rows** — the lines belonging to returned orders
- **558 returned orders** — matching `returns` exactly, so nothing was lost or
  duplicated
- **558 rows in `returns`** — the check that the flag found every one

If `returned ORDERS` came out below 558, some returns reference orders that are
not in silver, and that is a referential integrity failure worth escalating.

**The general rule: when you need a flag, flag. When you need columns, join.**
A merge whose only purpose is to set a boolean is a merge that can only hurt you.

### Question 6

Silver makes a promise, so test it. Write four assertions — `LineID` unique, no null join keys, no null dimension attributes after the joins, and row count equal to distinct bronze `LineID` — and print a PASS/FAIL line for each.
> **NOTE:** a layer whose promise is not checked is a layer whose promise is a hope.

In [ ]:
frames = [pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))]
bronze_orders = pd.concat(frames, ignore_index=True)
customers = pd.read_csv(BRONZE + "customers.csv")
products = pd.read_csv(BRONZE + "products.csv")
silver = (bronze_orders.drop_duplicates(subset="LineID", keep="first")
          .merge(customers[["CustomerID", "CustomerSegment", "Region"]],
                 on="CustomerID", how="left", validate="many_to_one")
          .merge(products[["ProductID", "ProductCategory"]],
                 on="ProductID", how="left", validate="many_to_one"))

CHECKS = [
    ("LineID is unique", True, bool(silver.LineID.is_unique)),
    ("no null CustomerID", 0, int(silver.CustomerID.isna().sum())),
    ("no null ProductID", 0, int(silver.ProductID.isna().sum())),
    ("no unmatched CustomerSegment", 0, int(silver.CustomerSegment.isna().sum())),
    ("no unmatched ProductCategory", 0, int(silver.ProductCategory.isna().sum())),
    ("rows == distinct bronze LineID",
     int(bronze_orders.LineID.nunique()), len(silver)),
]
failed = 0
for name, want, got in CHECKS:
    ok = want == got
    failed += not ok
    print("  %-32s expected %-6s got %-6s %s"
          % (name, want, got, "PASS" if ok else "FAIL"))
print()
print("%d of %d checks passed" % (len(CHECKS) - failed, len(CHECKS)))

```
  LineID is unique                 expected True   got True   PASS
  no null CustomerID               expected 0      got 0      PASS
  no null ProductID                expected 0      got 0      PASS
  no unmatched CustomerSegment     expected 0      got 0      PASS
  no unmatched ProductCategory     expected 0      got 0      PASS
  rows == distinct bronze LineID   expected 8060   got 8060   PASS

6 of 6 checks passed
```

Six assertions, one verdict. This is the gate between silver and anything that
reads it.

The checks are not interchangeable — each catches a different failure:

**`LineID is unique`** — the dedup ran. Catches a re-delivery that slipped through.

**No null join keys** — the source did not send unusable rows. A null key cannot
be joined and cannot be traced.

**No unmatched dimension attributes** — this is the important pair. `CustomerSegment`
and `ProductCategory` come from the *right* side of a left join, so a null there
means **the join found no match**. It is how a broken foreign key announces
itself, and it is invisible in the row count because a left join keeps the row
either way.

**`rows == distinct bronze LineID`** — the reconciliation, and the only check that
compares against something **outside** silver. The other five could all pass on a
silver table that had silently lost half its rows; this one could not. Expected
value derived from bronze, not read off silver.

Two things worth adding in a real pipeline:

**Trend checks.** Every expectation here is absolute. A source that halved
overnight passes all six. Comparing row counts and totals against the previous run
catches what absolutes cannot.

**Fail the pipeline, not the report.** These should run *before* gold is built and
stop it if they fail. A silver table that is known-bad and published anyway is
worse than one that is late.

GOLD — one question, one answer

### Question 7

Build a gold table: revenue by `CustomerSegment` and `ProductCategory`, with the line count, total sales, and returned sales. Print it.

In [ ]:
frames = [pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))]
customers = pd.read_csv(BRONZE + "customers.csv")
products = pd.read_csv(BRONZE + "products.csv")
returns = pd.read_csv(BRONZE + "returns.csv")
silver = (pd.concat(frames, ignore_index=True)
            .drop_duplicates(subset="LineID", keep="first")
            .merge(customers[["CustomerID", "CustomerSegment"]],
                   on="CustomerID", how="left", validate="many_to_one")
            .merge(products[["ProductID", "ProductCategory"]],
                   on="ProductID", how="left", validate="many_to_one"))
silver["IsReturned"] = silver.OrderID.isin(set(returns.OrderID))
silver["ReturnedSales"] = silver.Sales.where(silver.IsReturned, 0.0)

gold = (silver.groupby(["CustomerSegment", "ProductCategory"])
              .agg(lines=("LineID", "size"),
                   sales=("Sales", "sum"),
                   returned_sales=("ReturnedSales", "sum"))
              .reset_index())
gold["return_rate"] = (gold.returned_sales / gold.sales).round(4)
print(gold.to_string(index=False))
print()
print("rows:", len(gold), "| SUM(sales): %.2f" % gold.sales.sum())

```
CustomerSegment ProductCategory  lines        sales  returned_sales  return_rate
       Consumer       Furniture    330  991800.4740      81001.8540       0.0817
       Consumer Office Supplies    857  689040.7700      49599.5800       0.0720
       Consumer      Technology    397 1160619.2810      68369.6490       0.0589
      Corporate       Furniture    582 1595682.4340     145903.1500       0.0914
      Corporate Office Supplies   1665 1338391.0500     178048.8800       0.1330
      Corporate      Technology    700 2135875.0555     221803.3380       0.1038
    Home Office       Furniture    367 1006700.1600     102346.8400       0.1017
    Home Office Office Supplies   1142  959378.0900     116903.6500       0.1219
    Home Office      Technology    443 1225723.8845     119902.6040       0.0978
 Small Business       Furniture    297  732154.4180     123453.9820       0.1686
 Small Business Office Supplies    918  758737.7700     123975.2700       0.1634
 Small Business      Technology    362  976707.2415     154398.9345       0.1581

rows: 12 | SUM(sales): 13570810.63
```

**8,060 rows became 12**, and the total is unchanged at 13,570,810.63 — gold
reshapes, it does not lose.

That reconciliation is the check to run on every gold table: if the total moved,
a `groupby` dropped rows on a null key, or a filter crept in.

The design decision worth copying is `return_rate`. It is stored as a **ratio
derived from two stored measures**, not as a rate summed from somewhere:

```python
silver["ReturnedSales"] = silver.Sales.where(silver.IsReturned, 0.0)
gold["return_rate"] = gold.returned_sales / gold.sales
```

`sales` and `returned_sales` are both additive — they can be re-aggregated to any
coarser grain and stay correct. `return_rate` cannot: averaging the twelve rates
gives a different, wrong answer from `SUM(returned)/SUM(sales)`. **Store the
numerator and the denominator; derive the ratio.** Anyone rolling this up to
segment level must redo the division, and because both inputs are present, they
can.

And the numbers say something. **Small Business returns 15.8-16.9% of revenue
across all three categories** — roughly double Consumer's 5.9-8.2%, and
consistent across categories rather than driven by one. That consistency is what
makes it a finding rather than noise: three independent categories, same
direction, large gap.

Corporate Office Supplies is the largest cell by line count (1,665) and has a
13.3% return rate against Consumer Office Supplies' 7.2% — same products,
different buyers, double the returns. That is a question for the business, and
gold's job is to make it visible in twelve rows.

### Question 8

Build a second gold table at a different grain — revenue by `Region` — then show why you must not join the two gold tables. Print the row count of a merge between them on nothing in common, versus the correct way to get both answers.
> **NOTE:** two aggregates at different grains have no key to join on. Going back to silver is the answer, not joining golds.

In [ ]:
frames = [pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))]
customers = pd.read_csv(BRONZE + "customers.csv")
silver = (pd.concat(frames, ignore_index=True)
            .drop_duplicates(subset="LineID", keep="first")
            .merge(customers[["CustomerID", "CustomerSegment", "Region"]],
                   on="CustomerID", how="left", validate="many_to_one"))

by_segment = (silver.groupby("CustomerSegment")
                    .agg(sales=("Sales", "sum")).reset_index())
by_region = (silver.groupby("Region")
                   .agg(sales=("Sales", "sum")).reset_index())
print("gold by segment:", len(by_segment), "rows | total %.2f" % by_segment.sales.sum())
print("gold by region: ", len(by_region), "rows | total %.2f" % by_region.sales.sum())
print()
cross = by_segment.merge(by_region, how="cross", suffixes=("_seg", "_reg"))
print("cross join of the two golds:", len(cross), "rows")
print("SUM after cross join: %.2f  <- meaningless" % cross.sales_seg.sum())
print()
both = (silver.groupby(["CustomerSegment", "Region"])
              .agg(sales=("Sales", "sum")).reset_index())
print("from SILVER at the combined grain:", len(both),
      "rows | total %.2f" % both.sales.sum())

```
gold by segment: 4 rows | total 13570810.63
gold by region:  8 rows | total 13570810.63

cross join of the two golds: 32 rows
SUM after cross join: 108566485.03  <- meaningless

from SILVER at the combined grain: 32 rows | total 13570810.63
```

Look at the row counts: the wrong answer and the right answer are **both 32
rows**. The totals differ by a factor of eight — **108,566,485.03 against
13,570,810.63**.

That coincidence is the lesson. A row count is not evidence that a join was
correct. Two gold tables sharing no key produced a plausible-looking 4 x 8 grid,
and every number in it is fabricated: each segment's revenue repeated once per
region, each region's once per segment.

The cause is that **an aggregate has thrown away the key it would need to join
on**. `by_segment` has one row per segment; `by_region` one per region. Nothing
connects a segment row to a region row, because the connection lived at *line*
grain and was aggregated away. There is no correct join here — the operation is
not merely risky, it is undefined.

The fix is not a better join. It is to **go back to silver**:

```python
silver.groupby(["CustomerSegment", "Region"]).agg(sales=("Sales", "sum"))
```

32 rows, total 13,570,810.63, correct — because the grouping happens where both
attributes still exist on the same row.

Which gives the rule for gold tables:

> **Gold tables are leaves, not inputs.** Each is built from silver, and nothing
> is built from gold.

If a question needs two golds combined, it needs a third gold built from silver
at the combined grain. That feels wasteful and is not: recomputing from silver is
cheap and correct, whereas joining aggregates is cheap and wrong.

The related trap is joining two *fact* tables at different grains, which is the
same failure one layer down. The same answer applies: aggregate each to a common
grain first, then join — never the reverse.

### Question 9

Print the row count and `SUM(Sales)` at each layer — bronze, silver, and gold — as one table, so the pipeline reconciles end to end.
> **NOTE:** silver and gold must agree on the total. Bronze is allowed to differ, and you must be able to say by exactly how much and why.

In [ ]:
frames = [pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))]
customers = pd.read_csv(BRONZE + "customers.csv")
products = pd.read_csv(BRONZE + "products.csv")
bronze_orders = pd.concat(frames, ignore_index=True)
silver = (bronze_orders.drop_duplicates(subset="LineID", keep="first")
          .merge(customers[["CustomerID", "CustomerSegment"]],
                 on="CustomerID", how="left", validate="many_to_one")
          .merge(products[["ProductID", "ProductCategory"]],
                 on="ProductID", how="left", validate="many_to_one"))
gold = (silver.groupby(["CustomerSegment", "ProductCategory"])
              .agg(sales=("Sales", "sum")).reset_index())

print("%-10s %8s %16s" % ("LAYER", "ROWS", "SUM(Sales)"))
print("%-10s %8d %16.2f" % ("bronze", len(bronze_orders), bronze_orders.Sales.sum()))
print("%-10s %8d %16.2f" % ("silver", len(silver), silver.Sales.sum()))
print("%-10s %8d %16.2f" % ("gold", len(gold), gold.sales.sum()))
print()
print("silver == gold on total:",
      round(silver.Sales.sum(), 2) == round(gold.sales.sum(), 2))
print("bronze - silver: %.2f  (the %d re-delivered rows)"
      % (bronze_orders.Sales.sum() - silver.Sales.sum(),
         len(bronze_orders) - len(silver)))

```
LAYER          ROWS       SUM(Sales)
bronze         8100      13633504.89
silver         8060      13570810.63
gold             12      13570810.63

silver == gold on total: True
bronze - silver: 62694.27  (the 40 re-delivered rows)
```

Three lines that describe the whole pipeline, and they should be printed by every
run.

**Silver and gold agree exactly.** They must: gold is a reshaping of silver, so
any difference is a `groupby` dropping rows on a null key or a filter nobody
declared. This single equality is the strongest cheap check in the pipeline.

**Bronze differs, and the difference is fully explained.** 62,694.27 across
exactly 40 rows — the re-delivery from question 2. That is the distinction
between a reconciliation that *passes* and one that is merely *not investigated*:
the gap is not zero, and you can name it, size it, and point at the rows.

An unexplained gap of 62,694.27 would be an incident. The same gap with "the 40
re-delivered rows from 2011-12-24 to 2011-12-31" next to it is a documented,
expected property of the load.

Note the row counts also tell the story of what each layer is *for*: 8,100 rows
of raw delivery, 8,060 rows of business truth, 12 rows of answer. Each layer is
smaller and more opinionated than the last, and each is derivable from the one
before — so any of them can be rebuilt without going back to the source.

**Store this table.** Per run, per layer, row count and control total. It is a
few numbers, and it turns "did last night's load work?" from an investigation into
a glance — and turns "when did this number start being wrong?" into a query.

### Question 10

Finally, skip silver: aggregate revenue by segment **straight from bronze**, and assert it equals the silver-based figure. **This is supposed to fail.** Read the difference.

In [ ]:
frames = [pd.read_csv(p) for p in sorted(glob.glob(BRONZE + "orders_*.csv"))]
customers = pd.read_csv(BRONZE + "customers.csv")
bronze_orders = pd.concat(frames, ignore_index=True)

shortcut = (bronze_orders.merge(customers[["CustomerID", "CustomerSegment"]],
                                on="CustomerID", how="left")
            .groupby("CustomerSegment").Sales.sum())
proper = (bronze_orders.drop_duplicates(subset="LineID", keep="first")
          .merge(customers[["CustomerID", "CustomerSegment"]],
                 on="CustomerID", how="left")
          .groupby("CustomerSegment").Sales.sum())

print("straight from bronze:")
print(shortcut.round(2).to_string())
print()
print("via silver:")
print(proper.round(2).to_string())
print()
print("difference per segment:")
print((shortcut - proper).round(2).to_string())
print()
assert shortcut.round(2).equals(proper.round(2)), (
    "bronze-derived gold overstates revenue by %.2f -- silver was skipped"
    % (shortcut.sum() - proper.sum()))

```
straight from bronze:            via silver:
Consumer          2852423.12     Consumer          2841460.52
Corporate         5077631.51     Corporate         5069948.54
Home Office       3213717.68     Home Office       3191802.13
Small Business    2489732.58     Small Business    2467599.43

difference per segment:
Consumer          10962.60
Corporate          7682.97
Home Office       21915.55
Small Business    22133.15

AssertionError: bronze-derived gold overstates revenue by 62694.27 -- silver was skipped
```

**Every segment is overstated, and by different amounts.**

Look at the left column on its own, as a stakeholder would. Four segments, a
plausible ordering, Corporate largest at 5.08M, sensible magnitudes. There is
nothing to see. It is wrong by 62,694.27 in total and it looks completely fine.

The per-segment differences are the part that makes this genuinely dangerous.
They range from **7,682.97 (Corporate) to 22,133.15 (Small Business)** — not a
uniform bias you could spot as a scaling error, but a distortion that varies by
group, because the 40 re-delivered rows are not evenly distributed across
segments. So the *ranking* can shift, the *shares* shift, and any
segment-over-segment comparison inherits an arbitrary error.

And it is only 0.462% overall. Too small to trigger anything, big enough to be
wrong.

This is what silver is for. The shortcut is tempting precisely because it looks
harmless: one fewer step, one fewer table, same columns, same shape of answer.
What it skips is not "cleaning" in the abstract — it is the specific guarantee
that **one row means one thing**, without which every aggregate is counting an
unknown number of duplicates.

The assertion is the cheap defence, and note what makes it work: it compares
against a figure computed a **different way**. Any check that recomputes gold
from gold would pass here.

**What this sheet established:**

| | |
|---|---|
| bronze | 8,100 rows + lineage — `_source_file` and `_ingested_at`, no cleaning |
| the duplicates | **80 rows, 40 LineIDs, split across two files**, dated 2011-12-24→31, identical apart from lineage — a **re-delivery**, not corruption |
| silver | 8,060 rows; dropping 40 removes **62,694.27 (0.462%)** of phantom revenue |
| conforming | two `many_to_one` joins, **8,060 → 8,060 → 8,060** |
| the flag | `isin`, **0 rows changed**; a merge would have made it 837 |
| the gate | **6 of 6** checks, one of them reconciling against bronze |
| gold | 8,060 rows → **12**, total unchanged; Small Business returns **~16%** vs Consumer's **~6-8%** |
| joining two golds | 32 rows either way, and **108,566,485.03 instead of 13,570,810.63** |
| skipping silver | every segment overstated, by **7,682.97 to 22,133.15** — a distortion that varies by group |

**The three promises, and what breaks without each:**

- **Bronze — nothing is lost.** Without it you cannot prove what the source sent,
  or reproduce a problem after the fact.
- **Silver — one row means one thing.** Without it every count and every sum is
  over an unknown number of duplicates (question 10).
- **Gold — one question, one answer.** Without it two dashboards compute the same
  metric two ways and disagree.

Each layer is cheap to rebuild from the one before, which is the property that
makes the whole thing safe to fix when — not if — a rule turns out to be wrong.